# 01 · Dataset Exploration

**Round 1 walkthrough:** inspect the trip folders, count observations and road-network files, check time/coordinate quality, and create a first Folium map. The provided archive has training and student-test trips; evaluator-only data is not included.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
DATA=ROOT/"data"/"Student folder"
print("Project:",ROOT,"| data folder available:",DATA.exists())

Project: /Users/yashas/Documents/Codex/2026-11-18/referenced-chatgpt-conversation-this-is-an-2/ML_hackathon | data folder available: True


In [2]:
import pandas as pd
from pathsnap.eda import run
rows=run(DATA,ROOT/"results"/"dataset_inventory.csv")
inventory=pd.DataFrame(rows)
summary=inventory.groupby("partition").agg(trips=("trip_id","count"),gps_observations=("gps_rows","sum"),nodes=("node_rows","sum"),road_arcs=("arc_rows","sum"),reference_route_entries=("route_rows","sum"),invalid_rows_removed=("preprocessing_rows_removed","sum"),nonincreasing_timestamps=("nonincreasing_timestamp_steps","sum"),gaps_over_5s=("time_gap_over_5s_steps","sum"))
display(summary)

,trips,gps_observations,nodes,road_arcs,reference_route_entries,invalid_rows_removed,nonincreasing_timestamps,gaps_over_5s
partition,,,,,,,,
student_test,10,19160,90689,185041,3477,0,0,44
train,80,208417,1933485,3876904,31336,0,0,1216


## File formats and how they relate

- `.track`: longitude, latitude, timestamp in seconds.
- `.nodes`: node longitude and latitude; each row index is a node ID.
- `.arcs`: directed from-node and to-node IDs; each row index is an arc ID.
- `.route`: chronological reference arc IDs for the trip.

The GPS rows do not correspond one-to-one to node or arc rows. Each trip has its own road graph.

In [3]:
from pathsnap.data import read_trip
from pathsnap.matcher import generate_candidates
from pathsnap.visualize import create_map
from IPython.display import IFrame,display
trip=read_trip(DATA/"student_test"/"00000000","student_test")
batch=generate_candidates(trip,k=12)
nearest=batch.ids[range(len(batch.ids)),batch.features[:,:,0].argmin(axis=1)]
map_path=ROOT/"maps"/"phase1_eda_map.html"
create_map(trip,batch,nearest,map_path)
print("Trip:",trip.trip_id,"GPS rows:",len(trip.track),"nodes:",len(trip.nodes),"arcs:",len(trip.arcs),"reference arcs:",len(trip.route))
print("Candidate miss rate:",round(100*float(batch.candidate_miss.mean()),2),"%")
display(IFrame(src=str(map_path),width="100%",height=480))

Trip: 00000000 GPS rows: 2503 nodes: 8542 arcs: 18196 reference arcs: 179
Candidate miss rate: 9.79 %


The first map separates observed GPS, candidate road context, reference route, and nearest-road baseline into layers. It is a visual exploration of an actual student-test development trip.